# Técnicas de Score de modelos en Python

Las técnicas de Score (o puntuación) de modelos de ML evalúan su precisión. Habitualmente se basan en realizar algún tipo de comparación entre la salida obtenida y la salida deseada.

Hay muchas técnicas de Score en ML, pero una familia de técnicas muy populares son las basadas en las Confusion matrices. Estas matrices cruzan las predicciones posibles, con las salidas deseadas, estableciendo una clasificación de la salida del modelo como:
- TN: True Negative, el modelo predice correctamente un resultado negativo.
- FN: False Negative, el modelo predice incorrectamente un resultado negativo.
- TP: True Positive, el modelo predice correctamente un resultado positivo.
- FP: False Positive, el modelo predice incorrectamente un resultado positivo.

Las técnicas más comunes populares de este tipo son: precision, recall, f1-score y accuracy.

En Pytorch no se implementa nativamente ninguna técnica de Score. Pero, en el ecosistema de Pytorch, hay un librería estándar de-facto que se puede usar para ello: *torchmetrics*

Este librería tiene un API que implementa varias métricas con una interfaz común que proporciona los métodos: *update()*, *compute()* y *reset()*. Veamos un ejemplo completo de su uso:

In [1]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 9.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchmetrics] [torchmetrics]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
from torchmetrics import F1Score
from torch.utils.data import DataLoader, TensorDataset

# Definimos un modelo muy simple para clasificación binaria
class BinaryClassificationModel(nn.Module):
    def __init__(self):
        super(BinaryClassificationModel, self).__init__()
        self.linear = nn.Linear(10, 1)
    
    def forward(self, x):
        output = torch.sigmoid(self.linear(x))
        return output

# Inicializamos el modelo, la métrica y el optimizador
model = BinaryClassificationModel()
metric = F1Score(task='binary',num_classes=1, threshold=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Creamos un dataset de ejemplo aleatorio
inputs = torch.randn(100, 10)
targets = torch.randint(0, 2, (100,)).float()  # Binary targets
dataset = TensorDataset(inputs, targets)
dataloader = DataLoader(dataset, batch_size=10)

# Bucle de entrenamiento
for epoch in range(5):  # 5 épocas
    running_loss = 0.0
    for i, data in enumerate(dataloader, 0):
        inputs, labels = data
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = nn.BCELoss()(outputs.view(-1), labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        # Actualizamos la métrica
        metric(outputs.view(-1), labels)
        
    print(f"Epoch {epoch+1}, Loss: {running_loss / len(dataloader)}")
    print(f"Epoch {epoch+1}, F1-Score: {metric.compute()}")

# Reinicializamos la métrica para la siguiente época
metric.reset()


Epoch 1, Loss: 0.695664393901825
Epoch 1, F1-Score: 0.523809552192688
Epoch 2, Loss: 0.6928554952144623
Epoch 2, F1-Score: 0.523809552192688
Epoch 3, Loss: 0.6905372619628907
Epoch 3, F1-Score: 0.5179283022880554
Epoch 4, Loss: 0.6883216321468353
Epoch 4, F1-Score: 0.5165165066719055
Epoch 5, Loss: 0.686177271604538
Epoch 5, F1-Score: 0.5133171677589417


También podemos usar una estructura de tipo *MetricCollection* para recopilar dos métricas a la vez. Veamos una adaptación del ejemplo anterior

In [ ]:
from torchmetrics import HammingDistance, F1Score, MetricCollection
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Inicializamos el modelo y el optimizador
model = BinaryClassificationModel() # Asumo que lo tienes definido más arriba

# En versiones actuales de torchmetrics, debes especificar el tipo de tarea (task="binary")
metrics = MetricCollection([
    HammingDistance(task="binary"), 
    F1Score(task="binary")
])
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Creamos un dataset de ejemplo aleatorio
inputs = torch.randn(100, 10)
targets = torch.randint(0, 2, (100,)).long()  # Binary targets
dataset = TensorDataset(inputs, targets)
dataloader = DataLoader(dataset, batch_size=10)

# Bucle de entrenamiento
for epoch in range(5):  # 5 épocas
    running_loss = 0.0
    for i, data in enumerate(dataloader, 0):
        inputs, labels = data
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = nn.BCELoss()(outputs.view(-1), labels.float())
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
        # Actualizamos la métrica (¡Tu código aquí estaba perfecto!)
        pred_labels = torch.round(outputs.view(-1)).int() # Es buena práctica forzarlo a entero (.int())
        metrics(pred_labels, labels)

    # 🛠️ CORRECCIÓN 2: Los prints van FUERA del bucle interno, a la altura de la época
    print(f"Epoch {epoch+1}, Loss: {running_loss / len(dataloader)}")
    print(f"Epoch {epoch+1}, Metrics: {metrics.compute()}")
    
    # Reinicializamos la métrica para la siguiente época (¡Perfecto también!)
    metrics.reset()

Epoch 1, Loss: 0.7564593851566315
Epoch 1, Metrics: {'BinaryHammingDistance': tensor(0.5000), 'BinaryF1Score': tensor(0.4186)}
Epoch 2, Loss: 0.7525813221931458
Epoch 2, Metrics: {'BinaryHammingDistance': tensor(0.5000), 'BinaryF1Score': tensor(0.4186)}
Epoch 3, Loss: 0.7491957485675812
Epoch 3, Metrics: {'BinaryHammingDistance': tensor(0.5000), 'BinaryF1Score': tensor(0.4186)}
Epoch 4, Loss: 0.7459256768226623
Epoch 4, Metrics: {'BinaryHammingDistance': tensor(0.4900), 'BinaryF1Score': tensor(0.4235)}
Epoch 5, Loss: 0.7427440404891967
Epoch 5, Metrics: {'BinaryHammingDistance': tensor(0.5000), 'BinaryF1Score': tensor(0.4048)}


# Ejercicio

Utilizando como el base, el código resultante del ejercicio al final del notebook *optimizers.ipynb*, haz los cambios pertinentes para calcular la Hamming Distance y el BinaryF1Score.

In [ ]:
# Import libraries
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

# --- NUEVO: Importar las métricas ---
from torchmetrics import MetricCollection, HammingDistance, F1Score

# Load CIFAR-10 dataset
transform = transforms.Compose(
    [transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,
                                        shuffle=True, num_workers=2)

# Define the network architecture
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Define the loss function
loss_func = nn.CrossEntropyLoss()

# Define the training function
def train(net, trainloader, optimizer, num_epochs=6):
    loss_values = []
    
    # --- NUEVO: Inicializar las métricas para un problema de 10 clases ---
    metrics = MetricCollection([
        HammingDistance(task="multiclass", num_classes=10),
        F1Score(task="multiclass", num_classes=10)
    ])
    
    for epoch in range(num_epochs):  # loop over the dataset multiple times
        running_loss = 0.0
        for i, data in enumerate(trainloader, 0):
            # get the inputs; data is a list of [inputs, labels]
            inputs, labels = data

            # zero the parameter gradients
            optimizer.zero_grad()

            # forward + backward + optimize
            outputs = net(inputs)
            loss = loss_func(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            
            # --- NUEVO: Actualizar las métricas ---
            # Las 'outputs' son 10 probabilidades. Usamos argmax para quedarnos 
            # con el índice de la clase ganadora (ej. clase 3: 'gato')
            preds = torch.argmax(outputs, dim=1)
            metrics(preds, labels)

            if i % 2000 == 1999:  # print every 2000 mini-batches
                loss_values.append(running_loss / 2000)
                running_loss = 0.0
                
        # --- NUEVO: Calcular, imprimir y resetear métricas al final de la época ---
        epoch_metrics = metrics.compute()
        print(f"Epoch {epoch+1} Metrics: {epoch_metrics}")
        metrics.reset()
        
    return loss_values

# ----------------- ENTRENAMIENTO Y PLOT -----------------

print("--- Iniciando entrenamiento con SGD ---")
net_SGD = Net()
optimizer_SGD = optim.SGD(net_SGD.parameters(), lr=0.001, momentum=0.9)
loss_SGD = train(net_SGD, trainloader, optimizer_SGD)

print("\n--- Iniciando entrenamiento con Adam ---")
net_Adam = Net()
optimizer_Adam = optim.Adam(net_Adam.parameters(), lr=0.001)
loss_Adam = train(net_Adam, trainloader, optimizer_Adam)    

print("\n--- Iniciando entrenamiento con RMSprop ---")
net_RMSprop = Net()
optimizer_RMSprop = optim.RMSprop(net_RMSprop.parameters(), lr=0.001)
loss_RMSprop = train(net_RMSprop, trainloader, optimizer_RMSprop)

# Plotting
plt.figure(figsize=(12, 8))
plt.plot(loss_SGD, label='SGD')
plt.plot(loss_Adam, label='Adam')
plt.plot(loss_RMSprop, label='RMSprop')
plt.xlabel('Epochs (x2000 mini-batches)')
plt.ylabel('Loss')
plt.title('Loss Evolution with Different Optimizers')
plt.legend()
plt.show()

--- Iniciando entrenamiento con SGD ---
Epoch 1 Metrics: {'MulticlassHammingDistance': tensor(0.6157), 'MulticlassF1Score': tensor(0.3843)}
Epoch 2 Metrics: {'MulticlassHammingDistance': tensor(0.4801), 'MulticlassF1Score': tensor(0.5199)}
Epoch 3 Metrics: {'MulticlassHammingDistance': tensor(0.4277), 'MulticlassF1Score': tensor(0.5723)}
Epoch 4 Metrics: {'MulticlassHammingDistance': tensor(0.3945), 'MulticlassF1Score': tensor(0.6055)}
